In [1]:
# Notebook: nb_bronze_validation
# Purpose: Validate all Bronze Delta tables and log results

from pyspark.sql import functions as F
from datetime import datetime

print("🔍 Starting Bronze Layer Validation...")
print("=" * 60)
# Define the tables to validate and their key columns
bronze_tables = {
    "bronze_cipc_companies": ["RegistrationNumber", "CompanyName", "RegistrationDate"],
    "bronze_sanctions_entities": ["id", "name", "schema"],
    "bronze_jse_listings": ["JSE_Code", "CompanyName", "Sector"],
    "bronze_sarb_stats": ["BankName", "BankCode", "Quarter"]
}

# List to hold validation results
validation_results = []

StatementMeta(, 15afa29a-efc1-475a-88d9-094e74969f60, 3, Finished, Available, Finished, False)

🔍 Starting Bronze Layer Validation...


In [2]:
# Validate Each Bronze Table
# ==========================================
for table_name, key_columns in bronze_tables.items():
    print(f"\n Validating: {table_name}")
    print("-" * 40)
    
    try:
        # Read the Bronze table
        df = spark.table(table_name)
        
        # 1. Row Count
        row_count = df.count()
        print(f"  Total Rows: {row_count}")
        
        # 2. Null Check on Key Columns
        null_checks = {}
        for col in key_columns:
            if col in df.columns:
                null_count = df.filter(F.col(col).isNull()).count()
                null_pct = (null_count / row_count * 100) if row_count > 0 else 0
                null_checks[col] = null_pct
                status = "✅" if null_pct < 5 else "⚠️"
                print(f"  {status} {col} null rate: {null_pct:.2f}%")
        
        # 3. Date Range Check (if date columns exist)
        date_columns = [c for c in df.columns if 'date' in c.lower() or 'Date' in c]
        date_range = None
        if date_columns:
            for date_col in date_columns[:1]:  # Check first date column
                min_date = df.agg(F.min(F.col(date_col))).collect()[0][0]
                max_date = df.agg(F.max(F.col(date_col))).collect()[0][0]
                date_range = f"{min_date} to {max_date}"
                print(f"  📅 {date_col} range: {date_range}")
        
        # 4. Determine Pass/Fail Status
        max_null_rate = max(null_checks.values()) if null_checks else 0
        status = "PASS" if (row_count > 0 and max_null_rate < 10) else "FAIL"
        status_icon = "✅" if status == "PASS" else "❌"
        print(f"\n  {status_icon} Overall Status: {status}")
        
        # Store results
        validation_results.append({
            "table_name": table_name,
            "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "row_count": row_count,
            "null_rate_pct": round(max_null_rate, 2),
            "date_range": date_range if date_range else "N/A",
            "status": status
        })
        
    except Exception as e:
        print(f"  Error validating {table_name}: {str(e)}")
        validation_results.append({
            "table_name": table_name,
            "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "row_count": 0,
            "null_rate_pct": 100.0,
            "date_range": "N/A",
            "status": "FAIL"
        })


StatementMeta(, 15afa29a-efc1-475a-88d9-094e74969f60, 4, Finished, Available, Finished, False)


 Validating: bronze_cipc_companies
----------------------------------------
  Total Rows: 1000
  ✅ RegistrationNumber null rate: 0.00%
  ✅ CompanyName null rate: 0.00%
  ✅ RegistrationDate null rate: 0.00%
  📅 RegistrationDate range: 2015-01-02 to 2023-12-23

  ✅ Overall Status: PASS

 Validating: bronze_sanctions_entities
----------------------------------------
  Total Rows: 500
  ✅ id null rate: 0.00%
  ✅ name null rate: 0.00%
  ✅ schema null rate: 0.00%

  ✅ Overall Status: PASS

 Validating: bronze_jse_listings
----------------------------------------
  Total Rows: 50
  ✅ JSE_Code null rate: 0.00%
  ✅ CompanyName null rate: 0.00%
  ✅ Sector null rate: 0.00%

  ✅ Overall Status: PASS

 Validating: bronze_sarb_stats
----------------------------------------
  Total Rows: 40
  ✅ BankName null rate: 0.00%
  ✅ BankCode null rate: 0.00%
  ✅ Quarter null rate: 0.00%

  ✅ Overall Status: PASS


In [3]:
# Write Validation Results to Pipeline Logs Table
# ==========================================
print("\n" + "=" * 60)
print(" Writing validation results to bronze_pipeline_logs...")

results_df = spark.createDataFrame(validation_results)

# Write as Delta table (overwrite to keep only latest run)
results_df.write.format("delta").mode("overwrite").saveAsTable("bronze_pipeline_logs")

print(f" Successfully logged {len(validation_results)} validation results.")

StatementMeta(, 15afa29a-efc1-475a-88d9-094e74969f60, 5, Finished, Available, Finished, False)


 Writing validation results to bronze_pipeline_logs...
 Successfully logged 4 validation results.


In [4]:
# Display Summary

print("\n" + "=" * 60)
print(" VALIDATION SUMMARY")
print("=" * 60)
display(results_df)

# Final status check
all_passed = all(r["status"] == "PASS" for r in validation_results)
if all_passed:
    print("\n All Bronze tables passed validation!")
else:
    print("\n️  Some tables failed validation. Check logs for details.")

StatementMeta(, 15afa29a-efc1-475a-88d9-094e74969f60, 6, Finished, Available, Finished, False)


 VALIDATION SUMMARY


SynapseWidget(Synapse.DataFrame, cf0cfaba-3236-44c2-a6ac-79c8647980f1)


 All Bronze tables passed validation!
